<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 3 · Validación cruzada y métricas desde cero

**Unidad 3**

En clase vimos que el error dentro de la muestra, $E_{\text{in}}$, es una mala guía para elegir modelos, y que para estimar $E_{\text{out}}$ hay que guardar datos que el modelo no haya visto. También vimos que, en clasificación, «¿qué tan bueno es el modelo?» depende de qué errores nos importan más. En este laboratorio vas a escribir tú mismo las herramientas con las que se responden esas preguntas: la regresión *ridge* en forma cerrada, la validación cruzada con su regla de una desviación estándar, la curva ROC, el área bajo ella y la elección de un umbral de decisión según los costos.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** Dentro de tus funciones usa solo numpy (y `scipy.stats.rankdata` donde se indica): nada de `sklearn`, que aquí se usa únicamente para comparar. Evita los ciclos sobre los datos: las operaciones deben ser vectorizadas. Un ciclo sobre los pliegues o sobre los valores de $\lambda$ sí está permitido.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "main"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u03-lab03-validacion-metricas"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/main/unidades/03-fundamentos/laboratorios/tests/u03-lab03-validacion-metricas/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

import rdp

rdp.estilo.usar()

## Los datos y el problema

Usaremos los datos de **incumplimiento de pago de tarjetas de crédito** de un banco de Taiwán, publicados por I.-C. Yeh en el repositorio UCI (*Default of Credit Card Clients*, 2009, licencia CC BY 4.0). Cada fila es un cliente, con información de abril a septiembre de 2005:

- `limite`: línea de crédito (en nuevos dólares taiwaneses) y `edad`;
- `atraso_1` … `atraso_6`: situación de pago de cada mes (−1 pagó a tiempo; 1 a 9, meses de atraso), donde `_1` es septiembre y `_6` es abril;
- `factura_1` … `factura_6`: monto del estado de cuenta de cada mes;
- `pago_1` … `pago_6`: monto que pagó cada mes;
- `incumple`: 1 si el cliente **no pagó** en octubre. Es lo que queremos predecir.

**El problema.** El banco quiere un puntaje que ordene a sus clientes del más al menos riesgoso, y una regla para decidir a quién tratar como «probable incumplido» (por ejemplo, para reducirle la línea de crédito). Equivocarse en un sentido o en otro no cuesta lo mismo: dejar pasar a un cliente que no pagará cuesta mucho más que molestar a uno que sí pagaría.

El archivo tiene 30,000 clientes. Para que todo corra rápido trabajaremos con una **muestra estratificada fija de 6,000** (conserva la proporción de incumplidos) y con las 20 variables numéricas. Los montos tienen colas muy largas, así que los transformamos con $\operatorname{signo}(x)\log(1 + |x|)$.

In [ ]:
credito = rdp.datos.cargar("credito_taiwan")
muestra, _ = train_test_split(credito, train_size=6000, stratify=credito["incumple"], random_state=0)

columnas = [c for c in credito.columns if c not in ("incumple", "sexo", "escolaridad", "estado_civil")]
X_todo = muestra[columnas].to_numpy(dtype=float)
y_todo = muestra["incumple"].to_numpy(dtype=float)
montos = [i for i, c in enumerate(columnas) if c.startswith(("limite", "factura", "pago"))]
X_todo[:, montos] = np.sign(X_todo[:, montos]) * np.log1p(np.abs(X_todo[:, montos]))

# 4,000 clientes para entrenar y elegir modelos; 2,000 se guardan para el final
X_ent, X_pru, y_ent, y_pru = train_test_split(X_todo, y_todo, test_size=2000, stratify=y_todo, random_state=0)
media, escala = X_ent.mean(axis=0), X_ent.std(axis=0)
X_ent = (X_ent - media) / escala
X_pru = (X_pru - media) / escala

print(f"entrenamiento: {X_ent.shape} · prueba: {X_pru.shape} · incumplidos: {y_todo.mean():.1%}")

Antes de cualquier modelo, un número para tener en mente: el clasificador que dice **«nadie incumple»** acierta en el 78 % de los casos. Lo regresaremos a ver al final.

In [ ]:
print(f"exactitud de «nadie incumple» en prueba: {(y_pru == 0).mean():.1%}")

## 1. Regresión ridge en forma cerrada

Empezamos con el modelo más sencillo que da un puntaje de riesgo: una regresión lineal sobre la etiqueta 0/1. La predicción $\hat y = w_0 + x^\top w$ se puede leer como una estimación (burda) de la probabilidad de incumplir, y su error cuadrático medio es el **puntaje de Brier**.

Con 20 variables muy correlacionadas (las seis facturas se parecen mucho entre sí), los mínimos cuadrados ordinarios tienen varianza alta. La regresión **ridge** agrega una penalización que encoge los pesos:

$$\min_{w_0,\,w}\; \sum_{i=1}^{N} \bigl(y_i - w_0 - x_i^\top w\bigr)^2 + \lambda \lVert w \rVert^2 .$$

El **sesgo** $w_0$ no se penaliza: no tiene sentido encoger el nivel promedio de las predicciones hacia cero (si sumamos una constante a todas las $y_i$, la solución debería solo cambiar $w_0$). Derivando respecto a $w_0$ se obtiene $w_0 = \bar y - \bar x^\top w$, y sustituyendo queda un problema sobre los datos **centrados** $\tilde X = X - \bar x$, $\tilde y = y - \bar y$, cuya solución es

$$w = \bigl(\tilde X^\top \tilde X + \lambda I\bigr)^{-1} \tilde X^\top \tilde y, \qquad w_0 = \bar y - \bar x^\top w .$$

Para $\lambda > 0$ la matriz $\tilde X^\top \tilde X + \lambda I$ es definida positiva, así que siempre es invertible, aunque las columnas sean colineales.

**Pregunta 1.** Escribe `ridge(X, y, lam)`, que devuelve la tupla `(w0, w)`: el sesgo (un `float`) y el vector de pesos de longitud $d$, según las fórmulas de arriba. `X` es una matriz de $N \times d$ **sin** columna de unos y no necesariamente centrada. Usa `np.linalg.solve` en lugar de invertir la matriz.

In [ ]:
def ridge(X, y, lam):
    """Regresión ridge sin penalizar el sesgo. Devuelve (w0, w)."""
    ...


w0, w = ridge(X_ent, y_ent, 10.0)
print(f"w0 = {w0:.4f}; los tres pesos más grandes en valor absoluto:")
for j in np.argsort(-np.abs(w))[:3]:
    print(f"  {columnas[j]:>10}: {w[j]:+.4f}")

In [ ]:
grader.check("p1")

## 2. Validación cruzada

Para elegir $\lambda$ no podemos usar el error en entrenamiento: siempre es menor con $\lambda = 0$. En **validación cruzada con $k$ pliegues** partimos los índices $\{0, \dots, N-1\}$ en $k$ bloques $P_1, \dots, P_k$ disjuntos; para cada $j$ entrenamos con todo menos $P_j$ y medimos el error $E_j$ en $P_j$. La estimación es

$$E_{\text{cv}} = \frac{1}{k}\sum_{j=1}^{k} E_j ,$$

y la dispersión de los $E_j$ nos dice qué tan ruidosa es esa estimación.

**Pregunta 2.** Escribe `pliegues(n, k, semilla)`, que devuelve una **lista** de $k$ arreglos de enteros: los índices de prueba de cada pliegue. Hazlo así, para que todos obtengamos los mismos pliegues:

1. baraja los índices con `np.random.default_rng(semilla).permutation(n)`;
2. corta la permutación en $k$ bloques **consecutivos** cuyos tamaños difieran a lo más en 1, con los bloques más grandes al principio (eso hace `np.array_split`).

Los $k$ arreglos deben formar una **partición** de $\{0, \dots, n-1\}$: cada índice aparece en exactamente un pliegue.

In [ ]:
def pliegues(n, k, semilla=0):
    """Lista de k arreglos con los índices de prueba de cada pliegue."""
    ...


pliegues(10, 3, semilla=0)

In [ ]:
grader.check("p2")

Para que la validación cruzada sirva con cualquier modelo, la escribiremos en términos de dos funciones:

- `ajustar(X, y)` entrena un modelo y **devuelve una función** `predecir(X_nuevo)`;
- `metrica(y_real, y_predicha)` devuelve un número (menor es mejor).

Por ejemplo, el modelo que siempre predice la media de entrenamiento y el error cuadrático medio:

In [ ]:
def ecm(y_real, y_predicha):
    """Error cuadrático medio."""
    return float(np.mean((np.asarray(y_real) - np.asarray(y_predicha)) ** 2))


def ajustar_media(X, y):
    m = y.mean()
    return lambda X_nuevo: np.full(len(X_nuevo), m)


predecir = ajustar_media(X_ent, y_ent)
print(f"ECM en entrenamiento del modelo constante: {ecm(y_ent, predecir(X_ent)):.4f}")

**Pregunta 3.** Escribe `validacion_cruzada(ajustar, X, y, k, semilla, metrica)`. Usa tus `pliegues(len(y), k, semilla)`; para cada pliegue entrena con los demás índices, predice el pliegue y mide el error. Devuelve la tupla `(media, desviacion)`: el promedio de los $k$ errores y su **desviación estándar muestral** (`np.std(..., ddof=1)`).

In [ ]:
def validacion_cruzada(ajustar, X, y, k=10, semilla=0, metrica=ecm):
    """Error de validación cruzada: (media, desviación estándar muestral) de los k errores."""
    ...


validacion_cruzada(ajustar_media, X_ent, y_ent, k=10, semilla=0)

In [ ]:
grader.check("p3")

In [ ]:
_m, _s = validacion_cruzada(ajustar_media, X_ent, y_ent, k=10, semilla=0)  # deja esta celda como está
print(f"Modelo constante: ECM de validación cruzada = {_m:.4f} ± {_s:.4f}")

## 3. Elegir $\lambda$ con la regla de una desviación estándar

Ya tenemos todo para elegir $\lambda$: calculamos $E_{\text{cv}}(\lambda)$ en una rejilla de valores y nos quedamos con el mínimo. Pero $E_{\text{cv}}$ es una estimación ruidosa; si varios valores de $\lambda$ dan errores que no se distinguen dentro de ese ruido, preferimos el **modelo más sencillo**, el más regularizado. Esa es la **regla de una desviación estándar**:

1. sea $\lambda_{\min}$ el valor con menor $E_{\text{cv}}$, y $\text{ee} = s_{\min}/\sqrt{k}$ el **error estándar** de esa media ($s_{\min}$ es la desviación estándar de sus $k$ errores);
2. elige el **mayor** $\lambda$ de la rejilla con $E_{\text{cv}}(\lambda) \le E_{\text{cv}}(\lambda_{\min}) + \text{ee}$.

Abajo está `ajustar_ridge(lam)`, que convierte tu `ridge` en una función `ajustar` como la que pide `validacion_cruzada`.

In [ ]:
def ajustar_ridge(lam):
    def ajustar(X, y):
        w0, w = ridge(X, y, lam)
        return lambda X_nuevo: w0 + X_nuevo @ w
    return ajustar


rejilla = np.logspace(-2, 5, 29)  # de 0.01 a 100,000

**Pregunta 4.** Para cada $\lambda$ de `rejilla` (en ese orden) calcula la validación cruzada de ridge sobre `X_ent`, `y_ent` con `k=10`, `semilla=0` y `metrica=ecm`. Guarda:

- `errores_cv`: arreglo de 29 medias, y `desv_cv`: arreglo de 29 desviaciones estándar;
- `lam_min`: el $\lambda$ con menor error medio;
- `lam_1de`: el $\lambda$ elegido con la regla de una desviación estándar.

In [ ]:
errores_cv = ...
desv_cv = ...
lam_min = ...
lam_1de = ...
print(f"lam_min = {lam_min:,.1f} · lam_1de = {lam_1de:,.1f}")

In [ ]:
grader.check("p4")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ee = desv_cv / np.sqrt(10)
ax.fill_between(rejilla, errores_cv - ee, errores_cv + ee, color=rdp.estilo.AZUL, alpha=0.15, lw=0)
ax.plot(rejilla, errores_cv, color=rdp.estilo.AZUL, marker="o", markersize=4)
ax.axhline(errores_cv.min() + ee[np.argmin(errores_cv)], color=rdp.estilo.GRIS, ls="--")
for lam, nombre, color in [(lam_min, "mínimo", rdp.estilo.ROJO), (lam_1de, "una desv. estándar", rdp.estilo.DORADO_OSC)]:
    ax.axvline(lam, color=color, ls=":")
    ax.text(lam, ax.get_ylim()[1], f" {nombre}\n λ = {lam:,.0f}", color=color, va="top", ha="left")
ax.set_xscale("log")
ax.set_title("Validación cruzada de ridge (10 pliegues): media ± error estándar")
ax.set_xlabel("λ (escala log)")
ax.set_ylabel("ECM de validación (puntaje de Brier)");

A la izquierda la curva es plana: con 4,000 clientes y 20 variables, un poco de regularización casi no cambia nada. A la derecha, cuando $\lambda$ es enorme, los pesos se encogen tanto que el modelo se acerca al constante y el error sube. La regla de una desviación estándar se mueve hacia la derecha tanto como lo permite el ruido de la estimación.

## 4. La curva ROC

Ahora entrenamos dos modelos con **todo** el conjunto de entrenamiento: ridge con `lam_1de` y una regresión logística (de scikit-learn, porque aquí nos interesan las métricas, no el modelo). Los dos dan un **puntaje** por cliente en el conjunto de prueba; mientras más alto, más riesgoso.

In [ ]:
w0, w = ridge(X_ent, y_ent, lam_1de)
puntaje_ridge = w0 + X_pru @ w

logistica = LogisticRegression(max_iter=1000).fit(X_ent, y_ent)
prob_logistica = logistica.predict_proba(X_pru)[:, 1]

Un puntaje se convierte en clasificador con un **umbral** $u$: decimos «incumple» si el puntaje es $\ge u$. Cada umbral da una tasa de verdaderos positivos ($\text{TPR}$, la exhaustividad) y una de falsos positivos ($\text{FPR}$):

$$\text{TPR}(u) = \frac{\#\{i : y_i = 1,\ s_i \ge u\}}{\#\{i : y_i = 1\}}, \qquad \text{FPR}(u) = \frac{\#\{i : y_i = 0,\ s_i \ge u\}}{\#\{i : y_i = 0\}} .$$

La **curva ROC** recorre todos los umbrales, de mayor a menor. Con $u = +\infty$ nadie es positivo y estamos en $(0, 0)$; al bajar el umbral por debajo del menor puntaje todos son positivos y llegamos a $(1, 1)$. Solo hace falta un punto por cada **valor distinto** de los puntajes: si varios clientes empatan, al bajar el umbral entran todos a la vez, y la curva da un salto diagonal.

**Pregunta 5.** Escribe `curva_roc(y, puntajes)`, que devuelve la tupla `(fpr, tpr)` de arreglos con un punto inicial $(0, 0)$ y luego un punto por cada valor distinto de `puntajes`, usando esos valores como umbrales en orden **decreciente** (el último punto es $(1, 1)$). Debe coincidir con `roc_curve(y, puntajes, drop_intermediate=False)`.

Pista: ordena los puntajes de mayor a menor, acumula positivos y negativos con `np.cumsum` y quédate solo con la **última** posición de cada grupo de empates.

In [ ]:
def curva_roc(y, puntajes):
    """Curva ROC: (fpr, tpr), con un punto por cada valor distinto del puntaje, más el (0, 0)."""
    ...


curva_roc(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8]))

In [ ]:
grader.check("p5")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for puntaje, nombre, color in [(puntaje_ridge, "ridge", rdp.estilo.AZUL), (prob_logistica, "logística", rdp.estilo.ROJO)]:
    fpr, tpr = curva_roc(y_pru, puntaje)
    ax.plot(fpr, tpr, color=color, label=nombre)
ax.plot([0, 1], [0, 1], color=rdp.estilo.GRIS, ls="--")
ax.text(0.62, 0.55, "al azar", color=rdp.estilo.TENUE, rotation=42)
ax.legend(loc="lower right")
ax.set_aspect("equal")
ax.set_title("Curvas ROC en el conjunto de prueba")
ax.set_xlabel("tasa de falsos positivos")
ax.set_ylabel("tasa de verdaderos positivos");

## 5. El área bajo la curva, sin la curva

El área bajo la curva ROC (AUC) tiene una interpretación probabilística muy útil: si escogemos al azar un cliente que incumple, $P$, y uno que no, $Q$,

$$\text{AUC} = \Pr\bigl(s_P > s_Q\bigr) + \tfrac{1}{2}\Pr\bigl(s_P = s_Q\bigr).$$

Es decir, la fracción de **pares** (positivo, negativo) que el puntaje ordena bien, con los empates valiendo medio. Esta es la estadística $U$ de Mann–Whitney dividida entre $n_1 n_0$ (el número de pares). Contar los $n_1 n_0$ pares directamente cuesta $O(n^2)$, pero hay un atajo con **rangos**: si ordenamos todos los puntajes y $r_i$ es la posición de $s_i$ (de 1 a $n$, y a los empatados se les da el promedio de sus posiciones), la suma de los rangos de los positivos cuenta, para cada positivo, cuántos puntajes tiene por debajo. Restando los pares positivo–positivo, $n_1(n_1+1)/2$, queda

$$\text{AUC} = \frac{\sum_{i:\,y_i = 1} r_i \;-\; n_1(n_1+1)/2}{n_1\, n_0},$$

que cuesta lo que cuesta ordenar: $O(n \log n)$.

**Pregunta 6.** Escribe `auc_pares(y, puntajes)` con la fórmula de rangos. Puedes usar `rankdata` de scipy, que ya da a los empates el promedio de sus posiciones. Devuelve un `float`. Con $n = 30{,}000$ debe tardar unos milisegundos: no compares los pares uno a uno (ni con ciclos ni formando la matriz de $n_1 \times n_0$ diferencias). Después calcula `auc_ridge` y `auc_logistica` en el conjunto de prueba.

In [ ]:
def auc_pares(y, puntajes):
    """AUC por la estadística de Mann–Whitney, con rangos (los empates valen 1/2)."""
    ...


auc_ridge = ...
auc_logistica = ...
print(f"AUC ridge: {auc_ridge:.4f} · AUC logística: {auc_logistica:.4f}")

In [ ]:
grader.check("p6")

Una comprobación más de que entendimos qué mide el AUC: es el área bajo tu `curva_roc`, calculada con la regla del trapecio (los saltos diagonales de los empates son justo el «medio punto» de la fórmula de pares).

In [ ]:
fpr, tpr = curva_roc(y_pru, prob_logistica)
trapecio = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)
print(f"regla del trapecio: {trapecio:.6f} · fórmula de pares: {auc_logistica:.6f}")

## 6. Un umbral según los costos

El AUC resume todos los umbrales a la vez, pero el banco tiene que tomar **una** decisión por cliente. Supongamos que tratar como riesgoso a un cliente que sí pagaría (falso positivo) cuesta $c_{\text{FP}} = 1$ (se le molesta y quizá se pierda), y dejar pasar a uno que no pagará (falso negativo) cuesta $c_{\text{FN}} = 5$. El costo total de un umbral $u$ es

$$C(u) = c_{\text{FP}}\cdot \text{FP}(u) + c_{\text{FN}}\cdot \text{FN}(u).$$

Si el modelo diera la probabilidad **verdadera** $p$ de incumplir, la decisión óptima para un cliente sería marcarlo cuando el costo esperado de no hacerlo, $c_{\text{FN}}\,p$, supera al de hacerlo, $c_{\text{FP}}(1-p)$; es decir, cuando

$$p \;\ge\; \frac{c_{\text{FP}}}{c_{\text{FP}} + c_{\text{FN}}} = \frac{1}{6} \approx 0.167 .$$

El umbral 0.5 solo es óptimo cuando los dos errores cuestan lo mismo. Y como las probabilidades de un modelo nunca son exactamente las verdaderas, en la práctica el umbral se **elige con datos**.

In [ ]:
c_fp, c_fn = 1.0, 5.0
u_teorico = c_fp / (c_fp + c_fn)

**Pregunta 7.** Escribe `umbral_costo(y, prob, c_fp, c_fn)`, que devuelve la tupla `(umbral, costo)` con el umbral de menor costo total y ese costo. La regla es «positivo si `prob >= umbral`». Los umbrales candidatos son los **valores distintos** de `prob` más `np.inf` (que no marca a nadie); si varios empatan en costo, devuelve el **menor**. Hazlo vectorizado (con `np.unique`, `np.searchsorted` o `np.cumsum`), no con un ciclo sobre los candidatos.

Después, elige el umbral **sin tocar el conjunto de prueba**: abajo se calculan `prob_val`, probabilidades de la logística para cada cliente de entrenamiento obtenidas con validación cruzada (cada cliente recibe la probabilidad de un modelo que no lo vio). Guarda en `u_opt` el umbral que da `umbral_costo` con `y_ent` y `prob_val`.

In [ ]:
prob_val = np.empty(len(y_ent))
for indices in pliegues(len(y_ent), 5, semilla=0):
    entrena = np.setdiff1d(np.arange(len(y_ent)), indices)
    modelo = LogisticRegression(max_iter=1000).fit(X_ent[entrena], y_ent[entrena])
    prob_val[indices] = modelo.predict_proba(X_ent[indices])[:, 1]


def umbral_costo(y, prob, c_fp, c_fn):
    """Umbral (entre los valores de prob y +inf) que minimiza c_fp·FP + c_fn·FN. Devuelve (umbral, costo)."""
    ...


u_opt = ...
print(f"umbral elegido con validación cruzada: {u_opt:.3f} · teórico: {u_teorico:.3f}")

In [ ]:
grader.check("p7")

¿Cuánto cuesta cada regla en el conjunto de **prueba**, que nadie usó para decidir nada?

In [ ]:
def costo(y, prob, u):
    return c_fp * np.sum((prob >= u) & (y == 0)) + c_fn * np.sum((prob < u) & (y == 1))


umbrales = {"0.5 (por omisión)": 0.5, f"teórico ({u_teorico:.3f})": u_teorico, f"validación cruzada ({u_opt:.3f})": u_opt}
for nombre, u in umbrales.items():
    print(f"umbral {nombre:>26}: costo en prueba = {costo(y_pru, prob_logistica, u):,.0f}")
print(f"{'nadie incumple':>33}: costo en prueba = {costo(y_pru, prob_logistica, np.inf):,.0f}")

In [ ]:
u_grafica = np.linspace(0.01, 0.9, 200)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(u_grafica, [costo(y_ent, prob_val, u) / len(y_ent) for u in u_grafica], color=rdp.estilo.AZUL)
ax.plot(u_grafica, [costo(y_pru, prob_logistica, u) / len(y_pru) for u in u_grafica], color=rdp.estilo.ROJO)
rdp.estilo.etiquetar(ax, 0.91, {"validación cruzada": costo(y_ent, prob_val, 0.9) / len(y_ent),
                               "prueba": costo(y_pru, prob_logistica, 0.9) / len(y_pru)})
for u, nombre, color in [(u_opt, "elegido", rdp.estilo.DORADO_OSC), (0.5, "0.5", rdp.estilo.GRIS)]:
    ax.axvline(u, color=color, ls=":")
    ax.text(u, ax.get_ylim()[1], f" {nombre}", color=color, va="top")
ax.set_xlim(0, 1.12)
ax.set_xticks(np.arange(0, 1.0, 0.1))
ax.set_title("Costo promedio por cliente según el umbral ($c_{FP}=1$, $c_{FN}=5$)")
ax.set_xlabel("umbral sobre la probabilidad de incumplir")
ax.set_ylabel("costo por cliente");

## 7. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una.

<!-- BEGIN QUESTION -->

**Reflexión 1.** Calcula (en una celda de código aparte, si quieres) la exactitud de la logística en prueba con el umbral 0.5 y con `u_opt`, y compárala con la del clasificador «nadie incumple». ¿Por qué la exactitud es una mala métrica para este problema? ¿Qué métrica o métricas le recomendarías al banco para comparar modelos y para fijar la regla de decisión? Justifica con los números de esta libreta.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** Una compañera propone ahorrarse la validación cruzada de la pregunta 7: «elijo el umbral con `umbral_costo(y_pru, prob_logistica, ...)`, que da el menor costo en prueba, y reporto ese costo». ¿Qué tiene de malo? ¿Qué pasaría con el costo que reporta si el banco aplicara esa regla a clientes nuevos? Relaciónalo con la diferencia entre $E_{\text{in}}$ y $E_{\text{out}}$.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()